# ProteinBatchSuite: procesamiento por lotes reanudable

Este notebook ejecuta ColabFold para los FASTA de `ProteinBatchSuite/Input` y guarda los resultados en `ProteinBatchSuite/Output`. Está preparado para dímeros y tetrámeros representados con cadenas separadas por `:`.

## Uso
1. Abre este archivo en Google Colab y selecciona **Entorno de ejecución → Ejecutar todo** (*Runtime → Run all*).
2. Autoriza Google Drive cuando se solicite.
3. El lote procesa hasta `MAX_JOBS_PER_SESSION = 3` FASTA por ejecución.
4. Los ZIP se verifican antes de marcar una tarea como completada. Los resultados completos se omiten en ejecuciones posteriores.
5. Si Colab se desconecta, vuelve a abrir este notebook, conecta el entorno y ejecuta **Run all** otra vez.

Los archivos `batch_log.csv` y `batch_errors.csv` registran el progreso y los errores. Mantén una sola ejecución activa sobre esta carpeta de salida. La verificación del ZIP no garantiza que la estructura predicha sea biológicamente correcta.


In [ ]:
#@title Configuración de ProteinBatchSuite y Google Drive
from google.colab import drive
from pathlib import Path
from sys import version_info

python_version = f"{version_info.major}.{version_info.minor}"
drive.mount('/content/drive')

INPUT_FOLDER = Path("/content/drive/MyDrive/ProteinBatchSuite/Input")
OUTPUT_FOLDER = Path("/content/drive/MyDrive/ProteinBatchSuite/Output")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

if not INPUT_FOLDER.is_dir():
    raise FileNotFoundError(f"No existe la carpeta de entrada: {INPUT_FOLDER}")

fasta_count = sum(1 for p in INPUT_FOLDER.glob("*.fasta") if p.is_file())
if fasta_count == 0:
    raise FileNotFoundError(f"No se encontraron archivos .fasta en {INPUT_FOLDER}")

# Configuración que utiliza la instalación original de ColabFold.
num_relax = 0  #@param [0, 1, 5] {type:"raw"}
template_mode = "none"  #@param ["none", "pdb100"]
use_amber = num_relax > 0
use_templates = template_mode == "pdb100"
custom_template_path = None

print("Carpeta Input :", INPUT_FOLDER)
print("Carpeta Output:", OUTPUT_FOLDER)
print("FASTA encontrados:", fasta_count)
print("Relajación Amber:", num_relax)
print("Uso de templates:", use_templates)


In [ ]:
#@title Install dependencies
%%time
import os
USE_AMBER = use_amber
USE_TEMPLATES = use_templates
PYTHON_VERSION = python_version

if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  # hack to fix TF crash
  os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so /usr/local/lib/python3.*/dist-packages/tensorflow/lite/python/*/*.so")
  os.system("touch COLABFOLD_READY")

if USE_AMBER or USE_TEMPLATES:
  if not os.path.isfile("CONDA_READY"):
    print("installing conda...")
    os.system("wget -qnc https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh")
    os.system("bash Miniforge3-Linux-x86_64.sh -bfp /usr/local")
    os.system("mamba config --set auto_update_conda false")
    os.system("touch CONDA_READY")

if USE_TEMPLATES and not os.path.isfile("HH_READY") and USE_AMBER and not os.path.isfile("AMBER_READY"):
  print("installing hhsuite and amber...")
  os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
  os.system("touch HH_READY")
  os.system("touch AMBER_READY")
else:
  if USE_TEMPLATES and not os.path.isfile("HH_READY"):
    print("installing hhsuite...")
    os.system(f"mamba install -y -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python='{PYTHON_VERSION}'")
    os.system("touch HH_READY")
  if USE_AMBER and not os.path.isfile("AMBER_READY"):
    print("installing amber...")
    os.system(f"mamba install -y -c conda-forge openmm=8.2.0 python='{PYTHON_VERSION}' pdbfixer")
    os.system("touch AMBER_READY")

In [ ]:
#@markdown ### Opciones MSA y emparejamiento para procesamiento por lotes
# No se incluye MSA personalizado en esta versión porque requeriría un archivo .a3m
# individual asociado a cada FASTA.
msa_mode = "mmseqs2_uniref_env" #@param ["mmseqs2_uniref_env", "mmseqs2_uniref", "single_sequence"]
pair_mode = "unpaired_paired" #@param ["unpaired_paired", "paired", "unpaired"]


In [ ]:
#@markdown ### Advanced settings
model_type = "auto" #@param ["auto", "alphafold2_ptm", "alphafold2_multimer_v1", "alphafold2_multimer_v2", "alphafold2_multimer_v3", "deepfold_v1", "alphafold2"]
#@markdown - if `auto` selected, will use `alphafold2_ptm` for monomer prediction and `alphafold2_multimer_v3` for complex prediction.
#@markdown Any of the mode_types can be used (regardless if input is monomer or complex).
num_recycles = "3" #@param ["auto", "0", "1", "3", "6", "12", "24", "48"]
#@markdown - if `auto` selected, will use `num_recycles=20` if `model_type=alphafold2_multimer_v3`, else `num_recycles=3` .
recycle_early_stop_tolerance = "auto" #@param ["auto", "0.0", "0.5", "1.0"]
#@markdown - if `auto` selected, will use `tol=0.5` if `model_type=alphafold2_multimer_v3` else `tol=0.0`.
relax_max_iterations = 200 #@param [0, 200, 2000] {type:"raw"}
#@markdown - max amber relax iterations, `0` = unlimited (AlphaFold2 default, can take very long)
pairing_strategy = "greedy" #@param ["greedy", "complete"] {type:"string"}
#@markdown - `greedy` = pair any taxonomically matching subsets, `complete` = all sequences have to match in one line.
calc_extra_ptm = False #@param {type:"boolean"}
#@markdown - return pairwise chain iptm/actifptm

#@markdown #### Sample settings
#@markdown -  enable dropouts and increase number of seeds to sample predictions from uncertainty of the model.
#@markdown -  decrease `max_msa` to increase uncertainity
max_msa = "auto" #@param ["auto", "512:1024", "256:512", "64:128", "32:64", "16:32"]
num_seeds = 1 #@param [1,2,4,8,16] {type:"raw"}
use_dropout = False #@param {type:"boolean"}

num_recycles = None if num_recycles == "auto" else int(num_recycles)
recycle_early_stop_tolerance = None if recycle_early_stop_tolerance == "auto" else float(recycle_early_stop_tolerance)
if max_msa == "auto": max_msa = None

#@markdown #### Save settings
save_all = False #@param {type:"boolean"}
save_recycles = False #@param {type:"boolean"}
save_to_google_drive = False #@param {type:"boolean"}
#@markdown - Los resultados se guardan directamente en la carpeta Output de Google Drive.
dpi = 200 #@param {type:"integer"}
#@markdown - set dpi for image resolution

#@markdown Don't forget to hit `Runtime` -> `Run all` after updating the form.

In [ ]:
#@title ProteinBatchSuite — procesamiento por lotes reanudable

import os
import re
import csv
import shutil
import zipfile
import traceback
import hashlib
from pathlib import Path
from datetime import datetime

from colabfold.download import download_alphafold_params
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type

# ============================================================
# AJUSTES DEL LOTE
# ============================================================
# Procesa automáticamente toda la cola pendiente. Los ZIP validados se
# conservan en Drive, de modo que una nueva sesión puede reanudar el trabajo.

INPUT_FOLDER = Path("/content/drive/MyDrive/ProteinBatchSuite/Input")
OUTPUT_FOLDER = Path("/content/drive/MyDrive/ProteinBatchSuite/Output")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
BATCH_LOG = OUTPUT_FOLDER / "batch_log.csv"
ERROR_LOG = OUTPUT_FOLDER / "batch_errors.csv"

requested_model_type = model_type

def append_csv_row(path, row):
    exists = path.exists() and path.stat().st_size > 0
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)

def read_fasta_sequence(path):
    pieces = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith(">"):
            pieces.append(line)
    sequence = "".join(pieces).replace(" ", "")
    if not sequence:
        raise ValueError("El FASTA no contiene secuencia.")
    chains = sequence.split(":")
    if any(not chain for chain in chains):
        raise ValueError("Hay una cadena vacía entre separadores ':'.")
    allowed = set("ACDEFGHIKLMNPQRSTVWYBXZJUO")
    invalid = sorted(set("".join(chains).upper()) - allowed)
    if invalid:
        raise ValueError(f"Caracteres no válidos en la secuencia: {invalid}")
    return sequence.upper()

def zip_status(path):
    """Devuelve (completo, cantidad_PDB, cantidad_scores), comprobando integridad y ranks únicos."""
    if not path.is_file() or path.stat().st_size == 0:
        return False, 0, 0
    try:
        with zipfile.ZipFile(path, "r") as archive:
            if archive.testzip() is not None:
                return False, 0, 0
            names = archive.namelist()
            pdb_names = [n for n in names if n.lower().endswith(".pdb")]
            score_names = [n for n in names if "_scores_rank_" in n and n.lower().endswith(".json")]
            pdb_ranks = {m.group(1) for n in pdb_names if (m := re.search(r"_rank_(\d+)_", Path(n).name))}
            score_ranks = {m.group(1) for n in score_names if (m := re.search(r"_scores_rank_(\d+)_", Path(n).name))}
            # Exigimos cinco ranks distintos y que PDB/score cubran los mismos ranks.
            complete = (
                len(pdb_ranks) >= 5
                and len(score_ranks) >= 5
                and pdb_ranks == score_ranks
            )
            return complete, len(pdb_names), len(score_names)
    except (zipfile.BadZipFile, OSError):
        return False, 0, 0

def input_features_callback(input_features):
    # En el modo por lotes no mostramos gráficos para reducir uso de memoria/salida.
    return None

def prediction_callback(protein_obj, length, prediction_result, input_features, mode):
    # No-op intencional: no se muestran estructuras durante el lote.
    return None

# ============================================================
# PREPARAR TAREAS PENDIENTES Y PRIORIZAR LAS NO INTENTADAS
# ============================================================
all_fasta = sorted(p for p in INPUT_FOLDER.glob("*.fasta") if p.is_file())
if not all_fasta:
    raise FileNotFoundError(f"No se encontraron archivos .fasta en {INPUT_FOLDER}")

pending = []
for fasta_path in all_fasta:
    target_zip = OUTPUT_FOLDER / f"{fasta_path.stem}.result.zip"
    complete, _, _ = zip_status(target_zip)
    if not complete:
        pending.append(fasta_path)

# Para que un error persistente en los primeros archivos no bloquee los demás,
# se procesan primero los FASTA pendientes que todavía no aparecen en el registro.
attempted_files = set()
if BATCH_LOG.exists() and BATCH_LOG.stat().st_size > 0:
    try:
        with BATCH_LOG.open("r", newline="", encoding="utf-8") as handle:
            for row in csv.DictReader(handle):
                if row.get("file"):
                    attempted_files.add(row["file"])
    except Exception as exc:
        print(f"Aviso: no se pudo leer {BATCH_LOG.name}: {exc}")

unattempted = [p for p in pending if p.name not in attempted_files]
previously_attempted = [p for p in pending if p.name in attempted_files]
ordered_pending = unattempted + previously_attempted

print("=" * 68)
print("PROTEINBATCHSUITE — LOTE REANUDABLE")
print("=" * 68)
print(f"FASTA encontrados             : {len(all_fasta)}")
print(f"Resultados completos          : {len(all_fasta) - len(pending)}")
print(f"Pendientes                    : {len(pending)}")
print(f"Pendientes nunca intentados   : {len(unattempted)}")
print("Modo de procesamiento          : toda la cola pendiente")
print(f"Registro de trabajos          : {BATCH_LOG}")
print(f"Registro de errores           : {ERROR_LOG}")
print("=" * 68)

if not pending:
    print("No quedan modelos pendientes. No se ejecutó ninguna predicción.")
else:
    batch = ordered_pending
    completed_this_run = 0
    failed_this_run = 0
    last_success = None

    for index, fasta_path in enumerate(batch, start=1):
        # Cada iteración vuelve a guardar el resultado validado en Drive.
        # Si Colab se interrumpe, al reanudar se omitirán los ZIP completos.
        target_zip = OUTPUT_FOLDER / f"{fasta_path.stem}.result.zip"
        started = datetime.now().isoformat(timespec="seconds")
        safe_jobname = re.sub(r"\W+", "", fasta_path.stem)
        if not safe_jobname:
            safe_jobname = "protein_" + hashlib.sha1(fasta_path.name.encode()).hexdigest()[:10]
        # Añadir hash si la sanitización cambia el nombre, reduciendo riesgo de colisiones.
        if safe_jobname != fasta_path.stem:
            safe_jobname = f"{safe_jobname}_{hashlib.sha1(fasta_path.name.encode()).hexdigest()[:8]}"

        print("\n" + "-" * 68)
        print(f"Trabajo {index}/{len(batch)}: {fasta_path.name}")
        print("-" * 68)

        complete, pdb_count, score_count = zip_status(target_zip)
        if complete:
            print("Omitido: el ZIP ya está completo y verificado.")
            append_csv_row(BATCH_LOG, {
                "timestamp": started, "file": fasta_path.name,
                "status": "SKIPPED_COMPLETE", "pdb_count": pdb_count,
                "score_count": score_count, "zip_path": str(target_zip), "error": ""
            })
            continue

        temp_zip = OUTPUT_FOLDER / f"{fasta_path.stem}.result.zip.partial"
        try:
            query_sequence = read_fasta_sequence(fasta_path)
            job_dir = Path("/content") / safe_jobname
            if job_dir.exists():
                shutil.rmtree(job_dir)
            job_dir.mkdir(parents=True, exist_ok=True)

            queries_path = job_dir / f"{safe_jobname}.csv"
            with queries_path.open("w", encoding="utf-8") as handle:
                handle.write("id,sequence\n")
                handle.write(f"{safe_jobname},{query_sequence}\n")

            setup_logging(job_dir / "log.txt")
            queries, is_complex = get_queries(str(queries_path))
            current_model_type = set_model_type(is_complex, requested_model_type)
            download_alphafold_params(current_model_type, Path("."))

            use_cluster_profile = not ("multimer" in current_model_type and max_msa is not None)

            print(f"Cadenas en entrada : {query_sequence.count(':') + 1}")
            print(f"Longitud total      : {len(query_sequence.replace(':', ''))}")
            print(f"Modelo              : {current_model_type}")
            print(f"MSA                 : {msa_mode}")
            print("Predicción en curso...")

            current_results = run(
                queries=queries,
                result_dir=str(job_dir),
                use_templates=use_templates,
                custom_template_path=custom_template_path,
                num_relax=num_relax,
                msa_mode=msa_mode,
                model_type=current_model_type,
                num_models=5,
                num_recycles=num_recycles,
                relax_max_iterations=relax_max_iterations,
                recycle_early_stop_tolerance=recycle_early_stop_tolerance,
                num_seeds=num_seeds,
                use_dropout=use_dropout,
                model_order=[1, 2, 3, 4, 5],
                is_complex=is_complex,
                data_dir=Path("."),
                keep_existing_results=False,
                rank_by="auto",
                pair_mode=pair_mode,
                pairing_strategy=pairing_strategy,
                stop_at_score=float(100),
                prediction_callback=prediction_callback,
                dpi=dpi,
                zip_results=False,
                save_all=save_all,
                max_msa=max_msa,
                use_cluster_profile=use_cluster_profile,
                input_features_callback=input_features_callback,
                save_recycles=save_recycles,
                user_agent="colabfold/google-colab-main",
                calc_extra_ptm=calc_extra_ptm,
            )

            if temp_zip.exists():
                temp_zip.unlink()
            with zipfile.ZipFile(temp_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
                for file_path in job_dir.rglob("*"):
                    if file_path.is_file():
                        archive.write(file_path, arcname=file_path.relative_to(job_dir.parent).as_posix())

            complete, pdb_count, score_count = zip_status(temp_zip)
            if not complete:
                raise RuntimeError(
                    f"ZIP no validado: {pdb_count} PDB y {score_count} archivos de puntuación; "
                    "se requieren al menos cinco ranks únicos de cada tipo."
                )

            # Reemplazo atómico dentro de Drive una vez validado el archivo temporal.
            temp_zip.replace(target_zip)
            print(f"Guardado y verificado: {target_zip.name}")
            print(f"Tamaño ZIP: {target_zip.stat().st_size / (1024**2):.2f} MB")
            print(f"Modelos PDB: {pdb_count} | Archivos de puntuación: {score_count}")

            last_success = {
                "jobname": safe_jobname, "results": current_results,
                "queries": queries, "is_complex": is_complex,
                "model_type": current_model_type, "query_sequence": query_sequence
            }
            completed_this_run += 1
            append_csv_row(BATCH_LOG, {
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "file": fasta_path.name, "status": "COMPLETED",
                "pdb_count": pdb_count, "score_count": score_count,
                "zip_path": str(target_zip), "error": ""
            })

        except Exception as exc:
            failed_this_run += 1
            error_text = f"{type(exc).__name__}: {exc}"
            print(f"ERROR en {fasta_path.name}: {error_text}")
            traceback.print_exc()
            if temp_zip.exists():
                try:
                    temp_zip.unlink()
                except OSError:
                    pass
            append_csv_row(BATCH_LOG, {
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "file": fasta_path.name, "status": "FAILED",
                "pdb_count": 0, "score_count": 0,
                "zip_path": str(target_zip), "error": error_text
            })
            append_csv_row(ERROR_LOG, {
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "file": fasta_path.name, "error": error_text,
                "traceback": traceback.format_exc()
            })

    if last_success is not None:
        jobname = last_success["jobname"]
        results = last_success["results"]
        queries = last_success["queries"]
        is_complex = last_success["is_complex"]
        model_type = last_success["model_type"]
        query_sequence = last_success["query_sequence"]

    print("\n" + "=" * 68)
    print("RESUMEN DEL LOTE")
    print("=" * 68)
    print(f"Completados en esta ejecución : {completed_this_run}")
    print(f"Fallidos en esta ejecución    : {failed_this_run}")
    print(f"Pendientes al inicio           : {len(pending)}")
    print(f"Archivos seleccionados         : {len(batch)}")
    print("Los ZIP completos se omitirán en futuras ejecuciones.")
    print("La celda recorrió toda la cola pendiente disponible al inicio.")
    print("Si Colab se desconectó antes de terminar, ejecuta Runtime → Run all para reanudar.")
    print("=" * 68)


## Uso y resolución de problemas

1. Abre este notebook en Google Colab y selecciona **Runtime → Run all** (Entorno de ejecución → Ejecutar todo).
2. Autoriza el acceso a Google Drive cuando se solicite.
3. La primera ejecución instala dependencias y puede tardar; no interrumpas la instalación.
4. La celda de procesamiento ejecuta hasta tres FASTA por ejecución. Para modificar el límite, cambia `MAX_JOBS_PER_SESSION` en esa celda.
5. Los resultados se guardan en `ProteinBatchSuite/Output`. `batch_log.csv` registra todos los intentos y `batch_errors.csv` los errores.
6. Tras una desconexión, vuelve a abrir el notebook, conecta el entorno y ejecuta **Runtime → Run all**. Los ZIP que pasen la validación se omiten.

**Notas:** el modo por lotes usa MSA de MMseqs2 o secuencia única; no admite archivos MSA personalizados por FASTA en esta versión. La verificación comprueba integridad del ZIP y cinco ranks únicos de PDB y archivos de puntuación; no demuestra que la estructura predicha sea biológicamente correcta. Mantén una sola ejecución activa sobre `Output`.
